# 📊 SPARK TUTORIAL 02: DATAFRAMES AVANZADOS

## 🎯 **OBJETIVO**
Dominar operaciones avanzadas con DataFrames de Spark

## 📋 **CONTENIDO**
- Transformaciones complejas
- Funciones de agregación
- Joins y operaciones relacionales
- Window Functions
- Optimización y caching

---

## 🔧 **CONFIGURACIÓN INICIAL**


In [ ]:
# Importar librerías avanzadas
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, lit, coalesce, regexp_replace, split, explode,
    count, sum as spark_sum, avg, max as spark_max, min as spark_min,
    row_number, rank, dense_rank, lag, lead, first, last,
    date_format, year, month, dayofmonth, datediff, current_date
)
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

print("📚 Librerías avanzadas importadas correctamente")


In [ ]:
# Crear SparkSession optimizada para DataFrames
import socket
import os

# Detectar si estamos dentro de un contenedor Docker
def get_spark_master():
    try:
        hostname = socket.gethostname()
        if 'jupyter' in hostname or 'master' in hostname:
            return "spark://master:7077"  # Desde dentro del contenedor
        else:
            return "spark://localhost:7077"  # Desde fuera del contenedor
    except:
        return "local[*]"  # Fallback a modo local

spark_master_url = get_spark_master()
print(f"🔧 Conectando a: {spark_master_url}")

spark = SparkSession.builder \
    .appName("EducacionIT-Spark-DataFrames") \
    .master(spark_master_url) \
    .config("spark.executor.memory", "800m") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.instances", "2") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()

print("✅ SparkSession creada con optimizaciones")


## 📊 **PASO 1: CREAR DATOS DE EJEMPLO**

Vamos a crear DataFrames más complejos para demostrar operaciones avanzadas.


In [ ]:
# DataFrame de empleados con más información
empleados_data = [
    (1, "Juan Pérez", "IT", 50000, "2020-01-15", "Madrid"),
    (2, "María García", "Marketing", 45000, "2019-03-20", "Barcelona"),
    (3, "Carlos López", "IT", 55000, "2021-06-10", "Madrid"),
    (4, "Ana Martínez", "HR", 40000, "2018-11-05", "Valencia"),
    (5, "Luis Rodríguez", "IT", 60000, "2017-09-12", "Sevilla"),
    (6, "Laura Sánchez", "Marketing", 48000, "2022-02-28", "Barcelona"),
    (7, "Pedro González", "Sales", 42000, "2020-07-15", "Madrid"),
    (8, "Carmen Ruiz", "IT", 52000, "2021-04-03", "Bilbao"),
    (9, "Miguel Torres", "HR", 38000, "2019-12-10", "Valencia"),
    (10, "Isabel Díaz", "Sales", 46000, "2020-08-22", "Sevilla")
]

empleados_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("nombre", StringType(), True),
    StructField("departamento", StringType(), True),
    StructField("salario", IntegerType(), True),
    StructField("fecha_ingreso", StringType(), True),
    StructField("ciudad", StringType(), True)
])

df_empleados = spark.createDataFrame(empleados_data, empleados_schema)
df_empleados = df_empleados.withColumn("fecha_ingreso", col("fecha_ingreso").cast(DateType()))

print("👥 DataFrame empleados:")
df_empleados.show()


## 🔄 **PASO 2: TRANSFORMACIONES AVANZADAS**

Vamos a ver transformaciones más complejas con DataFrames.


In [ ]:
# Agregar columnas calculadas
df_transformado = df_empleados \
    .withColumn("años_antigüedad", 
               datediff(current_date(), col("fecha_ingreso")) / 365) \
    .withColumn("salario_categoria", 
               when(col("salario") >= 55000, "Alto")
               .when(col("salario") >= 45000, "Medio")
               .otherwise("Bajo")) \
    .withColumn("inicial_nombre", 
               split(col("nombre"), " ")[0])

print("📊 DataFrame con columnas calculadas:")
df_transformado.show()


## 🪟 **PASO 3: WINDOW FUNCTIONS**

Las Window Functions permiten realizar cálculos sobre grupos de filas relacionadas.


In [ ]:
# Ranking por departamento
window_departamento = Window.partitionBy("departamento").orderBy(col("salario").desc())

df_ranking = df_empleados.withColumn("rank_salario", rank().over(window_departamento)) \
                        .withColumn("dense_rank_salario", dense_rank().over(window_departamento)) \
                        .withColumn("row_number_salario", row_number().over(window_departamento))

print("🏆 Ranking de empleados por salario dentro de cada departamento:")
df_ranking.select("nombre", "departamento", "salario", 
                 "rank_salario", "dense_rank_salario", "row_number_salario").show()


## 🎯 **RESUMEN DEL TUTORIAL**

¡Felicitaciones! Has completado el tutorial de DataFrames avanzados.

### **📚 Conceptos avanzados aprendidos:**
- ✅ **Transformaciones complejas**: Columnas calculadas y condicionales
- ✅ **Window Functions**: Ranking y análisis por grupos
- ✅ **Funciones de agregación**: Estadísticas avanzadas
- ✅ **Optimización**: Configuración avanzada de Spark
- ✅ **Tipos de datos**: Manejo de esquemas complejos

### **🚀 Próximos pasos:**
1. **Tutorial 03**: Spark SQL
2. **Experimentar** con tus propios datos
3. **Optimizar** consultas complejas

---

**🎉 ¡Has dominado los DataFrames avanzados de Spark!**


In [ ]:
# Cerrar SparkSession
spark.stop()
print("🔒 SparkSession cerrada correctamente")
